# llama.cpp `tool_choice="required"` one-time compatibility finder

이 노트북은 **MMM 런타임에 들어가는 코드가 아닙니다. 딱 한 번 실행해서 known-good llama.cpp commit을 찾는 검증 도구**입니다.

검증 대상은 MMM의 실제 로컬 프로필과 동일한 모델입니다.

- model: `unsloth/Qwen3.5-9B-MTP-GGUF`
- GGUF: `Qwen3.5-9B-UD-Q4_K_XL.gguf`
- server: native `llama-server`
- tool mode: `--jinja`
- test: `tool_choice="required"`가 자유 prose로 빠지지 않고 **실제 `tool_calls`를 반환하는지**
- small prompt + MMM 로그와 비슷한 large prompt(약 44k chars)를 둘 다 검사

이 노트북은 후보 commit을 한 번 찾아 결과를 출력할 뿐이며, **모드 생성 실행 때마다 테스트하지 않습니다.**

현재 MMM pinned commit `1d2869c6e54d5003f3927a79efbca0fefa034a6d`은 실패 기준(bad)으로 검사합니다.
관련 llama.cpp regression의 직전 commit `94bc47f2807805ffdc1c5fbe5dce5cd2afdf3a97`을 첫 good 후보로 검사합니다.


In [ ]:
# @title 1. 환경/모델 설정
MODEL_REPO = "unsloth/Qwen3.5-9B-MTP-GGUF"
MODEL_FILE = "Qwen3.5-9B-UD-Q4_K_XL.gguf"

# MMM 현재 pinned commit (실패 로그가 나온 엔진)
KNOWN_BAD = "1d2869c6e54d5003f3927a79efbca0fefa034a6d"

# 관련 regression #26398의 첫 bad(0ef6e55e...) 직전 commit.
# 이 commit이 실제 Qwen3.5 required-tool test를 통과하는지 이 노트북이 직접 확인합니다.
FIRST_GOOD_CANDIDATE = "94bc47f2807805ffdc1c5fbe5dce5cd2afdf3a97"

# candidate가 실패할 경우 first-parent history를 이 간격으로 뒤로 탐색합니다.
BACKTRACK_OFFSETS = [1, 2, 4, 8, 16, 32, 64, 128]

CTX_SIZE = 32768
PORT = 18931
SOURCE_DIR = "/content/llama.cpp-required-tool-bisect"
MODEL_DIR = "/content/mmm-models"

print("KNOWN_BAD:", KNOWN_BAD)
print("FIRST_GOOD_CANDIDATE:", FIRST_GOOD_CANDIDATE)


In [ ]:
# @title 2. 빌드 도구 + 모델 다운로드 (모델은 한 번만)
import os, sys, subprocess
from pathlib import Path

def run(cmd, *, cwd=None, check=True):
    print("+", " ".join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

run(["apt-get", "update"])
run([
    "apt-get", "install", "-y", "--no-install-recommends",
    "git", "cmake", "ninja-build", "ccache", "curl"
])
run([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "requests"])

from huggingface_hub import hf_hub_download

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir=MODEL_DIR,
)
print("MODEL_PATH =", MODEL_PATH)


In [ ]:
# @title 3. llama.cpp clone + CUDA 확인
import os, shutil, subprocess
from pathlib import Path

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("GPU runtime이 아닙니다. Colab에서 T4/L4/A100 GPU runtime으로 바꿔 실행하세요.")

print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"], text=True))

src = Path(SOURCE_DIR)
if not (src / ".git").is_dir():
    if src.exists():
        shutil.rmtree(src)
    run([
        "git", "clone", "--filter=blob:none",
        "https://github.com/ggml-org/llama.cpp.git", str(src)
])

run(["git", "-C", str(src), "fetch", "origin", "--tags", "--prune"])
run(["git", "-C", str(src), "fetch", "origin", KNOWN_BAD])
run(["git", "-C", str(src), "fetch", "origin", FIRST_GOOD_CANDIDATE])

# ccache를 모든 candidate build에서 재사용
os.environ["CCACHE_DIR"] = "/content/.ccache-llama-required-tool"
os.environ["CCACHE_MAXSIZE"] = "8G"
run(["ccache", "-M", os.environ["CCACHE_MAXSIZE"]])


In [ ]:
# @title 4. candidate build / server / required-tool test 함수
import json, os, signal, socket, subprocess, time, requests, shutil
from pathlib import Path

SRC = Path(SOURCE_DIR)
BUILD = SRC / "build-required-tool"

def git(*args):
    return subprocess.check_output(["git", "-C", str(SRC), *args], text=True).strip()

def checkout(ref):
    run(["git", "-C", str(SRC), "checkout", "--detach", "-f", ref])
    return git("rev-parse", "HEAD")

def configure_and_build(ref):
    sha = checkout(ref)
    env = os.environ.copy()
    env["CMAKE_C_COMPILER_LAUNCHER"] = "ccache"
    env["CMAKE_CXX_COMPILER_LAUNCHER"] = "ccache"
    run([
        "cmake", "-S", str(SRC), "-B", str(BUILD), "-G", "Ninja",
        "-DCMAKE_BUILD_TYPE=Release",
        "-DGGML_CUDA=ON",
        "-DGGML_CUDA_GRAPHS=ON",
        "-DGGML_CUDA_FA=ON",
        "-DLLAMA_BUILD_TESTS=OFF",
        "-DLLAMA_BUILD_EXAMPLES=OFF",
        "-DLLAMA_BUILD_APP=OFF",
        "-DLLAMA_BUILD_UI=OFF",
        "-DLLAMA_BUILD_SERVER=ON",
    ], check=True)
    subprocess.run(
        ["cmake", "--build", str(BUILD), "--target", "llama-server", "-j", str(min(8, os.cpu_count() or 1))],
        check=True, env=env,
    )
    binary = BUILD / "bin" / "llama-server"
    if not binary.is_file():
        raise RuntimeError(f"llama-server build missing at {binary}")
    return sha, binary

def wait_ready(proc, port, timeout=240):
    deadline = time.time() + timeout
    base = f"http://127.0.0.1:{port}"
    last = None
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"llama-server exited early: {proc.returncode}")
        try:
            r = requests.get(base + "/health", timeout=1)
            if r.status_code == 200:
                return base
            last = f"HTTP {r.status_code}"
        except Exception as e:
            last = repr(e)
        time.sleep(0.5)
    raise RuntimeError(f"server not ready: {last}")

TOOL = [{
    "type": "function",
    "function": {
        "name": "probe_tool",
        "description": "Return the requested project-relative path.",
        "parameters": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "path": {"type": "string", "minLength": 1}
            },
            "required": ["path"]
        }
    }
}]

def required_request(base, prompt):
    payload = {
        "model": "local",
        "messages": [{"role": "user", "content": prompt}],
        "tools": TOOL,
        "tool_choice": "required",
        "parallel_tool_calls": False,
        "temperature": 0.0,
        "max_tokens": 256,  # 검증 시간을 제한할 뿐 production 정책에 쓰지 않음
        "chat_template_kwargs": {"enable_thinking": False},
        "reasoning_effort": "none",
        "stream": False,
    }
    started = time.time()
    r = requests.post(base + "/v1/chat/completions", json=payload, timeout=180)
    elapsed = time.time() - started
    r.raise_for_status()
    data = r.json()
    choice = (data.get("choices") or [{}])[0]
    msg = choice.get("message") or {}
    calls = msg.get("tool_calls") or []
    fn = ""
    if calls:
        fn = ((calls[0].get("function") or {}).get("name") or "")
    usage = data.get("usage") or {}
    timings = data.get("timings") or {}
    completion_tokens = int(usage.get("completion_tokens") or timings.get("predicted_n") or 0)
    return {
        "ok": len(calls) == 1 and fn == "probe_tool",
        "finish_reason": choice.get("finish_reason"),
        "tool_calls": len(calls),
        "tool_name": fn,
        "completion_tokens": completion_tokens,
        "elapsed_s": round(elapsed, 2),
        "content_tail": str(msg.get("content") or "")[-120:],
    }

def evaluate_commit(ref):
    sha, binary = configure_and_build(ref)
    cmd = [
        str(binary),
        "-m", MODEL_PATH,
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--ctx-size", str(CTX_SIZE),
        "--batch-size", "2048",
        "--ubatch-size", "512",
        "--gpu-layers", "all",
        "--flash-attn", "on",
        "--cache-type-k", "q8_0",
        "--cache-type-v", "q8_0",
        "--jinja",
        "--no-ui",
        "--log-disable",
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    try:
        base = wait_ready(proc, PORT)
        props = requests.get(base + "/props", timeout=5).json()
        caps = props.get("chat_template_caps") or {}
        small = required_request(
            base,
            "Say hello. Do not use any tools. The server must enforce tool_choice=required anyway."
        )
        filler = ("Existing Java evidence line. " * 1700)[:44000]
        large = required_request(
            base,
            filler + "\nDo not use tools. Say hello."
        )
        ok = bool(small["ok"] and large["ok"])
        result = {
            "sha": sha,
            "ok": ok,
            "small": small,
            "large": large,
            "caps": {
                "supports_tool_calls": caps.get("supports_tool_calls"),
                "supports_preserve_reasoning": caps.get("supports_preserve_reasoning"),
            },
        }
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return result
    finally:
        proc.terminate()
        try:
            proc.wait(timeout=20)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)


In [ ]:
# @title 5. 먼저 현재 bad와 첫 good 후보를 실제 검사
RESULTS = {}

def test_once(ref):
    sha = git("rev-parse", ref)
    if sha in RESULTS:
        return RESULTS[sha]
    result = evaluate_commit(ref)
    RESULTS[sha] = result
    return result

bad_result = test_once(KNOWN_BAD)
candidate_result = test_once(FIRST_GOOD_CANDIDATE)

print("\nCURRENT PIN PASS?", bad_result["ok"])
print("FIRST GOOD CANDIDATE PASS?", candidate_result["ok"])

if bad_result["ok"]:
    print("현재 pinned commit이 이 독립 재현에서는 통과했습니다. MMM 요청 payload/프롬프트 차이를 별도로 비교해야 합니다.")
elif candidate_result["ok"]:
    print("GOOD/BAD bracket 확보:", candidate_result["sha"], "..", bad_result["sha"])
else:
    print("첫 good 후보도 실패합니다. 다음 셀에서 first-parent history를 뒤로 탐색합니다.")


In [ ]:
# @title 6. good commit이 없으면 과거로 지수 backtrack
def first_parent_at(ref, offset):
    revs = subprocess.check_output(
        ["git", "-C", str(SRC), "rev-list", "--first-parent", f"--max-count={offset+1}", ref],
        text=True
    ).splitlines()
    if len(revs) <= offset:
        return None
    return revs[offset]

GOOD_SHA = candidate_result["sha"] if candidate_result["ok"] else None
BAD_SHA = bad_result["sha"] if not bad_result["ok"] else None

if GOOD_SHA is None:
    base = candidate_result["sha"]
    for offset in BACKTRACK_OFFSETS:
        sha = first_parent_at(base, offset)
        if not sha:
            break
        print(f"\nBacktrack offset={offset}: {sha}")
        result = test_once(sha)
        if result["ok"]:
            GOOD_SHA = sha
            # candidate itself is known bad here; bracket is good..candidate
            BAD_SHA = candidate_result["sha"]
            break

if GOOD_SHA:
    print("GOOD bracket endpoint:", GOOD_SHA)
    print("BAD bracket endpoint :", BAD_SHA)
else:
    print("아직 good commit을 찾지 못했습니다. BACKTRACK_OFFSETS를 더 늘린 뒤 이 셀만 다시 실행하세요.")


In [ ]:
# @title 7. bracket이 있으면 binary search로 마지막 good / 첫 bad 정확히 찾기
def ancestry_path(good, bad):
    # good 이후 bad까지의 ancestry-path를 오래된 순서로 반환
    out = subprocess.check_output(
        ["git", "-C", str(SRC), "rev-list", "--ancestry-path", "--reverse", f"{good}..{bad}"],
        text=True
    ).splitlines()
    return [good] + out

FIRST_BAD_SHA = None
LAST_GOOD_SHA = GOOD_SHA

if GOOD_SHA and BAD_SHA:
    path = ancestry_path(GOOD_SHA, BAD_SHA)
    lo = 0                    # known good index
    hi = len(path) - 1        # known bad index
    if path[hi] != BAD_SHA:
        raise RuntimeError("BAD_SHA is not on the ancestry path from GOOD_SHA")
    while hi - lo > 1:
        mid = (lo + hi) // 2
        sha = path[mid]
        print(f"\nBisect {mid}/{len(path)-1}: {sha}")
        result = test_once(sha)
        if result["ok"]:
            lo = mid
        else:
            hi = mid
    LAST_GOOD_SHA = path[lo]
    FIRST_BAD_SHA = path[hi]
    print("\nLAST_GOOD_SHA =", LAST_GOOD_SHA)
    print("FIRST_BAD_SHA =", FIRST_BAD_SHA)
else:
    print("GOOD/BAD bracket이 없어 bisect를 실행하지 않습니다.")


In [ ]:
# @title 8. 최종 추천 SHA 출력 + 결과 저장
from pathlib import Path
import json

summary = {
    "model_repo": MODEL_REPO,
    "model_file": MODEL_FILE,
    "known_bad_input": KNOWN_BAD,
    "last_good_sha": LAST_GOOD_SHA,
    "first_bad_sha": FIRST_BAD_SHA,
    "results": RESULTS,
}

out = Path("/content/llama-required-tool-bisect-results.json")
out.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("=" * 80)
if LAST_GOOD_SHA:
    print("PIN THIS LLAMA.CPP COMMIT:")
    print(LAST_GOOD_SHA)
    if FIRST_BAD_SHA:
        print("\nFIRST BAD COMMIT:")
        print(FIRST_BAD_SHA)
else:
    print("known-good commit을 아직 찾지 못했습니다.")
print("\nRESULT JSON:", out)
print("=" * 80)

# 이 SHA를 MMM의 LLAMA_SERVER_SOURCE_REF와 native bundle build workflow에 고정하면 됩니다.
# 이 노트북 자체는 production runtime에서 호출하지 않습니다.
